# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 27.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=b52cb352-0313-4e95-85b3-5dae47b11f38
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:01<00:00, 21.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 164MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta.uniform_(-eps, eps)
        delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x+delta).detach()

def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        noise = torch.randn_like(x)
        noise_norm = l2_norm(noise).clamp_min(1e-12)
        radius = torch.rand(x.shape[0],1,1,1,device=x.device)
        delta = noise / noise_norm * radius * eps
        delta = clip_box(x,delta)
    for _ in range(steps):
        grad = input_grad(model,x,delta,y)
        grad_norm = l2_norm(grad).clamp_min(1e-12)
        delta = delta + alpha * grad / grad_norm
        delta_norm = l2_norm(delta).clamp_min(1e-12)
        scale = torch.clamp(eps/delta_norm, max=1.0)
        delta = delta * scale
        delta = clip_box(x, delta)
    return (x+delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.4  union 22.3   (114 s)


{'clean': 78.7, 'linf': 48.4, 'l2': 59.8, 'l1': 22.4, 'union': 22.3}

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.203 (40 s)
epoch 2: loss 1.180 (81 s)
epoch 3: loss 1.151 (122 s)
epoch 4: loss 1.126 (163 s)
epoch 5: loss 1.097 (204 s)
epoch 6: loss 1.068 (244 s)
epoch 7: loss 1.049 (285 s)
epoch 8: loss 1.030 (326 s)
epoch 9: loss 1.013 (367 s)
epoch 10: loss 1.008 (408 s)
clean 78.5  linf 44.9  l2 56.9  l1 20.6  union 20.5   (123 s)


{'clean': 78.5, 'linf': 44.9, 'l2': 56.9, 'l1': 20.6, 'union': 20.5}

In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.897 (41 s)
epoch 2: loss 0.825 (81 s)
epoch 3: loss 0.787 (122 s)
epoch 4: loss 0.754 (163 s)
epoch 5: loss 0.714 (204 s)
epoch 6: loss 0.675 (244 s)
epoch 7: loss 0.631 (285 s)
epoch 8: loss 0.609 (326 s)
epoch 9: loss 0.591 (367 s)
epoch 10: loss 0.583 (408 s)
clean 82.6  linf 32.1  l2 60.1  l1 27.8  union 24.3   (124 s)


{'clean': 82.6, 'linf': 32.1, 'l2': 60.1, 'l1': 27.8, 'union': 24.3}

In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.231 (41 s)
epoch 2: loss 0.838 (83 s)
epoch 3: loss 0.564 (124 s)
epoch 4: loss 0.449 (165 s)
epoch 5: loss 0.359 (206 s)
epoch 6: loss 0.277 (247 s)
epoch 7: loss 0.216 (289 s)
epoch 8: loss 0.186 (330 s)
epoch 9: loss 0.170 (371 s)
epoch 10: loss 0.160 (412 s)
clean 84.6  linf 0.0  l2 0.0  l1 27.5  union 0.0   (123 s)


{'clean': 84.6, 'linf': 0.0, 'l2': 0.0, 'l1': 27.5, 'union': 0.0}

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q = model(x_q)
    logits_r = model(x_r)
    loss_q = F.cross_entropy(logits_q, y, reduction="none")
    loss_r = F.cross_entropy(logits_r, y, reduction="none")
    adv_loss = torch.maximum(loss_q, loss_r).mean()
    p_q = F.softmax(logits_q, dim=1)
    log_p_r = F.log_softmax(logits_r, dim=1)
    kl_each = F.kl_div(log_p_r, p_q, reduction="none").sum(dim=1)
    correct_q = logits_q.argmax(dim=1) == y
    kl_loss = (kl_each * correct_q).sum() / correct_q.sum().clamp_min(1)
    return adv_loss + lam * kl_loss


def gradient_projection(g_n, g_a):
    cosine = F.cosine_similarity(g_n.reshape(-1), g_a.reshape(-1), dim=0)
    return cosine.clamp_min(0) * g_n

In [ ]:
model_ramp = load_base_model()
start = time.time()
model_ramp = adv_train(model_ramp, ramp_loss, gp_beta=0.5)
ramp_time_0 = (time.time()-start)/10
ramp_acc_0 = test_model(model_ramp)

model_ramp_1 = load_base_model()
start = time.time()
model_ramp_1 = adv_train(model_ramp_1, ramp_loss, gp_beta=0.5, seed=1)
ramp_time_1 = (time.time()-start)/10
ramp_acc_1 = test_model(model_ramp_1)

epoch 1: loss 1.431 (88 s)
epoch 2: loss 1.330 (175 s)
epoch 3: loss 1.303 (263 s)
epoch 4: loss 1.281 (351 s)
epoch 5: loss 1.254 (438 s)
epoch 6: loss 1.227 (526 s)
epoch 7: loss 1.199 (614 s)
epoch 8: loss 1.186 (701 s)
epoch 9: loss 1.171 (789 s)
epoch 10: loss 1.165 (876 s)
clean 79.0  linf 41.2  l2 61.6  l1 44.6  union 39.3   (124 s)
epoch 1: loss 1.433 (88 s)
epoch 2: loss 1.333 (175 s)
epoch 3: loss 1.299 (263 s)
epoch 4: loss 1.275 (350 s)
epoch 5: loss 1.248 (438 s)
epoch 6: loss 1.221 (525 s)
epoch 7: loss 1.200 (613 s)
epoch 8: loss 1.186 (700 s)
epoch 9: loss 1.178 (788 s)
epoch 10: loss 1.169 (876 s)
clean 79.2  linf 42.2  l2 62.6  l1 47.0  union 40.9   (124 s)


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [9]:
base_model = load_base_model()
base_model.eval()
def my_loss(model, x, y, lam=1.0):
    model.eval()
    x_linf = attack(model, x, y, "linf")
    x_l1 = attack(model, x, y, "l1")
    model.train()
    loss_l1 = F.cross_entropy(model(x_l1), y)
    with torch.no_grad():
        p_base = F.softmax(base_model(x_linf), dim=1)
    log_p = F.log_softmax(model(x_linf), dim=1)
    loss_keep = F.kl_div(log_p, p_base, reduction="batchmean")
    return loss_l1 + lam * loss_keep

In [ ]:
# model_mine = adv_train(load_base_model(), my_loss)
# test_model(model_mine)

In [10]:
model_mine_0 = load_base_model()
start = time.time()
model_mine_0 = adv_train(model_mine_0, my_loss, seed=0)
mine_time_0 = (time.time()-start)/10
mine_acc_0 = test_model(model_mine_0)
model_mine_1 = load_base_model()
start = time.time()
model_mine_1 = adv_train(model_mine_1, my_loss, seed=1)
mine_time_1 = (time.time()-start)/10
mine_acc_1 = test_model(model_mine_1)

epoch 1: loss 1.360 (86 s)
epoch 2: loss 1.170 (168 s)
epoch 3: loss 1.099 (251 s)
epoch 4: loss 1.041 (333 s)
epoch 5: loss 0.991 (415 s)
epoch 6: loss 0.939 (498 s)
epoch 7: loss 0.895 (580 s)
epoch 8: loss 0.869 (663 s)
epoch 9: loss 0.849 (745 s)
epoch 10: loss 0.841 (827 s)
clean 80.3  linf 35.1  l2 60.4  l1 51.9  union 34.9   (121 s)
epoch 1: loss 1.350 (82 s)
epoch 2: loss 1.182 (165 s)
epoch 3: loss 1.105 (247 s)
epoch 4: loss 1.041 (330 s)
epoch 5: loss 0.985 (412 s)
epoch 6: loss 0.927 (494 s)
epoch 7: loss 0.894 (577 s)
epoch 8: loss 0.867 (659 s)
epoch 9: loss 0.850 (741 s)
epoch 10: loss 0.839 (824 s)
clean 79.9  linf 35.6  l2 61.0  l1 53.1  union 35.2   (122 s)


In [11]:
mine_mean = {k: round((mine_acc_0[k]+mine_acc_1[k])/2, 2) for k in mine_acc_0}

print("mine:", mine_mean, f"{(mine_time_0+mine_time_1)/2:.1f} s/epoch")

mine: {'clean': 80.1, 'linf': 35.35, 'l2': 60.7, 'l1': 52.5, 'union': 35.05} 82.8 s/epoch


In [12]:
def no_base_loss(model, x, y):
    model.eval()
    x_linf = attack(model, x, y, "linf")
    x_l1 = attack(model, x, y, "l1")
    model.train()
    loss_l1 = F.cross_entropy(model(x_l1), y)
    loss_linf = F.cross_entropy(model(x_linf), y)
    return loss_l1 + loss_linf

model_no_base = adv_train(load_base_model(), no_base_loss, seed=0)
test_model(model_no_base)

epoch 1: loss 2.672 (82 s)
epoch 2: loss 2.490 (162 s)
epoch 3: loss 2.414 (243 s)
epoch 4: loss 2.322 (323 s)
epoch 5: loss 2.232 (404 s)
epoch 6: loss 2.138 (485 s)
epoch 7: loss 2.037 (565 s)
epoch 8: loss 1.971 (646 s)
epoch 9: loss 1.914 (727 s)
epoch 10: loss 1.892 (807 s)
clean 78.1  linf 40.2  l2 60.7  l1 48.4  union 39.2   (121 s)


{'clean': 78.1, 'linf': 40.2, 'l2': 60.7, 'l1': 48.4, 'union': 39.2}

In [13]:
if mine_acc_0["union"] >= mine_acc_1["union"]:
    best_model = model_mine_0
    best_default_acc = mine_acc_0
else:
    best_model = model_mine_1
    best_default_acc = mine_acc_1
strong_acc = test_model(best_model, steps=200, restarts=5)
print("default:", best_default_acc)
print("strong:", strong_acc)

clean 79.9  linf 35.4  l2 60.8  l1 49.0  union 34.8   (1210 s)
default: {'clean': 79.9, 'linf': 35.6, 'l2': 61.0, 'l1': 53.1, 'union': 35.2}
strong: {'clean': 79.9, 'linf': 35.4, 'l2': 60.8, 'l1': 49.0, 'union': 34.8}
